<a href="https://colab.research.google.com/github/prateekdhawan/FDE/blob/main/Assignment_3A_RAG/001.%20Agentic%20Router%20submission.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Deep Dive Agentic Retrieval Augmented Generation

An Agentic RAG is required when we use reasoning to determine which action(s) to take and in which order to take them. Essentially we use agents instead of a LLM directly to accomplish a set of tasks which requires planning, multi step reasoning, tool use and/or learning over time. Agents give us agency!

Agency : The ability to take action or to choose what action to take

In the context of RAG, we can plug in agents to enhance the reasoning prior to selection of RAG pipelines, within a RAG pipeline for retrieval or reranking and finally for synthesising before we send out the response. This improves RAG to a large extent by automating complex workflows and decisions that are required for a non trivial RAG use case.

### Purpose of this Agentic RAG
This notebook presents a practical implementation of Agentic Retrieval-Augmented Generation (RAG)—a system where decision-making and tool selection are delegated to an intelligent agent before executing a response. Rather than passing every query through a static RAG pipeline, this system introduces agency—the ability to choose the best course of action depending on the nature of the query.

At the heart of this implementation is a router prompt, which classifies user queries into one of three categories:

- OpenAI documentation: Queries related to tools, APIs, or usage guidelines for OpenAI models
- 10-K financial reports: Questions requiring retrieval from company filings or financial datasets
- Live Internet search: Broader, current, or comparative queries that need web access

Once the query is classified, the system invokes a corresponding route handler:

- For OpenAI and 10-K queries, it retrieves relevant context from a vector database (Qdrant) using text embeddings, then applies a RAG-based response generator.
- For Internet queries, it fetches real-time information using a web-access API (ARES).

This approach is an example of Agentic RAG, where reasoning precedes retrieval and generation. By plugging in agents before and within the RAG pipeline, we make the system smarter and more adaptive. This allows us to:

- Automatically choose the right retrieval method based on context
- Combine structured knowledge with real-time search
- Scale RAG beyond trivial use cases by integrating multi-step decision logic

Importantly, no external agentic frameworks are used—this is a ground-up implementation that demonstrates how to build a lightweight but intelligent agentic system using only a language model, prompt engineering, and retrieval tools.

## Setup and Dependencies

# Install the necessary libraries
!pip install openai
!pip install qdrant_client
!pip install transformers==4.48.0
!pip install einops==0.8.1   # required by nomic-embed-text-v1.5's trust_remote_code path

In [38]:
# Install the necessary libraries
!pip install openai
!pip install qdrant_client
!pip install transformers==4.48.0

In [39]:
# Import basic libraries
import requests             # Used for making HTTP requests (e.g., calling ARES API for live internet queries)
import json                 # For parsing and structuring JSON data (especially OpenAI and routing responses)

# Credentials — Colab Secrets when on Colab, a local .env otherwise
try:
    from google.colab import userdata          # Colab: keys live in the 🔑 Secrets panel
    IN_COLAB = True
except ImportError:                            # Local Jupyter: keys live in .env
    from dotenv import load_dotenv, find_dotenv
    load_dotenv(find_dotenv())
    IN_COLAB = False

    class userdata:                            # same .get() call works in both places
        @staticmethod
        def get(name):
            import os
            return os.getenv(name) or os.getenv(name.lower()) or os.getenv(
                name.replace("SERP_API_KEY", "SERPAPI_KEY"))

# OS operations
import os                   # Useful for accessing environment variables and managing paths

# OpenAI API client
from openai import OpenAI   # Official OpenAI client library to interface with GPT models for routing and generation

# Text processing
import re                   # Regular expressions for cleaning or preprocessing inputs (if needed)

# Optional visualization (for analysis/debugging purposes)
import matplotlib.pyplot as plt       # For displaying charts or visual debug outputs (e.g., embeddings visualizations)
import matplotlib.image as mpimg      # For loading/displaying images if needed (rare in RAG, but helpful in demos)

# Embedding models (used for text vectorization during retrieval)
from transformers import AutoTokenizer, AutoModel  # For loading custom transformer models if not using OpenAI embeddings

from qdrant_client import models

import qdrant_client
import asyncio
import nest_asyncio # Import nest_asyncio
nest_asyncio.apply() # Apply nest_asyncio to allow nested event loops
# # Vector database client
# from qdrant_client import QdrantClient   # Qdrant is used as the vector store to retrieve documents based on similarity

## 1. Defining the Internet Tool

First, we will define a tool function that enables our system to answer queries requiring real-time, internet-based information. Not all questions can be answered using static documents like OpenAI docs or financial filings—sometimes users ask about current trends, comparisons, or live updates.

To handle this, we introduce a live search capability using the **SerpApi**.

### What is SerpApi?  
SerpApi is a Google Search API that allows you to:

- Search the internet in real time using Google.
- Get structured results including answer boxes, organic results, and snippets.

This is particularly useful for questions about:

- Current events (e.g., *"Latest AI tools in 2025"*),
- Tech comparisons (e.g., *"Gemini vs GPT-4"*),
- General knowledge outside internal datasets.

Please generate the API key [here](https://serpapi.com)


In [40]:
#loads serp api key from colab secrets
serp_api_key=userdata.get('SERP_API_KEY')

In [41]:
import requests  # For sending HTTP requests to the SerpApi

def get_internet_content(user_query: str, action: str):
    """
    Fetches a response from the internet using SerpApi based on the user's query.

    This function serves as the tool invoked when the router classifies a query
    as requiring real-time information beyond internal datasets—i.e., "INTERNET_QUERY".
    It sends the query to SerpApi (Google Search) and returns structured results.

    Args:
        user_query (str): The user's question that needs a live answer.
        action (str): Route type (always expected to be "INTERNET_QUERY").

    Returns:
        str: Response text from live Google search results or an error message.
    """
    print("Getting your response from the internet 🌐 ...")

    params = {
        "q": user_query,
        "api_key": serp_api_key,
        "engine": "google",
        "num": 5,
    }

    try:
        response = requests.get("https://serpapi.com/search.json", params=params)
        response.raise_for_status()
        data = response.json()

        parts = []

        # Answer box — Google's highlighted direct answer (most relevant)
        answer_box = data.get("answer_box", {})
        if answer_box.get("answer"):
            parts.append(f"[Direct Answer] {answer_box['answer']}")
        elif answer_box.get("snippet"):
            parts.append(f"[Direct Answer] {answer_box['snippet']}")

        # Top organic results — titles + snippets
        for i, result in enumerate(data.get("organic_results", [])[:5], start=1):
            title = result.get("title", "")
            snippet = result.get("snippet", "")
            link = result.get("link", "")
            if snippet:
                parts.append(f"[{i}] {title}\n    {snippet}\n    Source: {link}")

        if not parts:
            return "No results found."

        return "\n\n".join(parts)

    # Handle HTTP-level errors (e.g., 400s or 500s)
    except requests.exceptions.HTTPError as http_err:
        return f"HTTP error occurred: {http_err}"

    # Handle general connection, timeout, or request formatting issues
    except requests.exceptions.RequestException as req_err:
        return f"Request error occurred: {req_err}"

    # Catch-all for any unexpected failure
    except Exception as err:
        return f"An unexpected error occurred: {err}"


In [42]:
print(get_internet_content("Tell me about best travel destinations in 2026?","INTERNET_QUERY")) #run internet function to test results

Getting your response from the internet 🌐 ...
[1] 10 most romantic destinations to consider exploring in 2026
    Kayak to hidden lagoons in Thailand · 2. Beneath the waters in Belize · 3. Experience endless ways to fall in love in the British Virgin Islands.
    Source: https://www.moorings.com/uk/blog/most-romantic-sailing-destinations

[2] 2026 plans and destinations? - Forums
    Japan is top of the list, Japan is top of the list, some Baltic and Central Europe too. Estonia, Latvia, and Lithuania, maybe part of Poland.
    Source: https://choosefi.com/forums/living-abroad/2026-plans-and-destinations

[3] From Zermatt to Les Arcs: Europe's top 10 ski resorts for ...
    Europe's best ski resorts for 2026 include picks in Switzerland, France and Austria, each offering experiences on and off the slopes and ...
    Source: https://www.euronews.com/travel/2026/08/10/from-zermatt-to-les-arcs-europes-top-10-ski-resorts-for-2026-according-to-tripcom

[4] We made one of the top destinations

## 2. Router Query Function — Giving the Agent Its Brain

In this step, we will define the router function, which plays a critical role in our Agentic RAG system.

### What is a Router?

A router is like the decision-making brain of our assistant.

Before trying to answer a user's question, the system first needs to figure out:

> “Where should I go to find the right answer?”

To make this decision, we use the OpenAI GPT model. We provide it with a detailed system prompt that explains how to classify the user's question into one of these categories:

- **OPENAI_QUERY** → Questions about OpenAI tools, APIs, models, or documentation.
- **10K_DOCUMENT_QUERY** → Questions about companies, financial filings, or analysis based on 10-K reports.
- **INTERNET_QUERY** → Anything else that likely requires real-time or general web information.

### What does the function do?

- Sends the user's question to the OpenAI API.
- Receives a JSON response containing:
  - `action`: The category the query belongs to.
  - `reason`: A short explanation for the decision.
  - `answer`: (Optional) A quick response if it’s simple enough (left blank for internet queries).
- Parses the response and returns it as a Python dictionary.

### Why is this important?

This router gives the system agency—the ability to decide which knowledge source to use. It’s what makes this pipeline agentic, not just static.

Without the router, every query would follow the same path. With it, we can:

- Dynamically switch between tools and data sources.
- Handle different types of user questions intelligently.
- Avoid wasting resources on unnecessary steps.


## Query Routing Workflow

The diagram below shows the full decision flow — from receiving a user query to returning a final response.

```
                        ┌─────────────────────┐
                        │     User Query      │
                        └──────────┬──────────┘
                                   │
                                   ▼
                    ┌──────────────────────────────┐
                    │      Router LLM (GPT-4o)     │
                    │         route_query()         │
                    │                              │
                    │  Reads the query and decides │
                    │  which data source to use    │
                    └──────────────┬───────────────┘
                                   │
           ┌───────────────────────┼───────────────────────┐
           │                       │                       │
           ▼                       ▼                       ▼
┌─────────────────────┐ ┌─────────────────────┐ ┌─────────────────────┐
│    OPENAI_QUERY     │ │ 10K_DOCUMENT_QUERY  │ │   INTERNET_QUERY    │
│                     │ │                     │ │                     │
│ e.g. "What are      │ │ e.g. "What was      │ │ e.g. "Best LLMs     │
│  OpenAI Agents?"    │ │  Uber's revenue?"   │ │  in 2026?"          │
└──────────┬──────────┘ └──────────┬──────────┘ └──────────┬──────────┘
           │                       │                        │
           ▼                       ▼                        ▼
┌─────────────────────┐ ┌─────────────────────┐  ┌──────────────────────┐
│   Embed Query       │ │   Embed Query        │  │      SerpApi         │
│  (Nomic Model)      │ │  (Nomic Model)       │  │  get_internet_       │
│                     │ │                      │  │  content()           │
│  get_text_          │ │  get_text_           │  │                      │
│  embeddings()       │ │  embeddings()        │  │  Live Google search  │
└──────────┬──────────┘ └──────────┬──────────┘  └──────────┬───────────┘
           │                       │                          │
           ▼                       ▼                          │
┌─────────────────────┐ ┌─────────────────────┐              │
│  Qdrant Vector DB   │ │  Qdrant Vector DB   │              │
│  Collection:        │ │  Collection:         │              │
│  "opnai_data"       │ │  "10k_data"          │              │
│                     │ │                      │              │
│  Retrieve top-3     │ │  Retrieve top-3      │              │
│  similar chunks     │ │  similar chunks      │              │
└──────────┬──────────┘ └──────────┬──────────┘              │
           │                       │                          │
           └───────────┬───────────┘                          │
                       ▼                                      │
           ┌───────────────────────┐                          │
           │    RAG Response       │                          │
           │    Generator          │                          │
           │  rag_formatted_       │                          │
           │  response()           │                          │
           │                       │                          │
           │  GPT-4 synthesizes    │                          │
           │  answer from context  │                          │
           │  + adds citations     │                          │
           └───────────┬───────────┘                          │
                       │                                      │
                       └──────────────────┬───────────────────┘
                                          ▼
                             ┌────────────────────────┐
                             │     Final Response     │
                             │       to User          │
                             └────────────────────────┘
```

### Key decision points at a glance

| Route | Trigger | Retrieval Method | Response Generator |
|---|---|---|---|
| `OPENAI_QUERY` | OpenAI docs, APIs, Agents | Qdrant `opnai_data` (top-3 chunks) | `rag_formatted_response()` via GPT-4 |
| `10K_DOCUMENT_QUERY` | Financial filings, company revenue | Qdrant `10k_data` (top-3 chunks) | `rag_formatted_response()` via GPT-4 |
| `INTERNET_QUERY` | Anything else / real-time info | SerpApi live Google search | Raw search result snippets |

> **Note:** Both vector-based routes share the same embedding model (`nomic-embed-text-v1.5`) and RAG generator — only the Qdrant collection changes. The router's JSON output (`action` field) is the single decision variable that drives the entire flow.


In [43]:
# ── LLM provider ─────────────────────────────────────────────────────────────
# This notebook was written for OpenAI (model alias "gpt-5.6-luna"), but we run it on
# Google Gemini's OpenAI-COMPATIBLE endpoint (free tier) — no OpenAI key needed. The
# `openai` SDK talks to Gemini unchanged; only the base_url + model name differ, and each
# single prompt is sent as a "user" message (Gemini 400s on a lone "system" message:
# "contents is not specified"). Embeddings stay LOCAL (Nomic, below) to match the
# prebuilt 768-dim Qdrant vectors, so no embedding key is required.
#
# Colab: add GOOGLE_API_KEY in the 🔑 Secrets panel. Local: it's in .env (git-ignored).
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
google_api_key = userdata.get('GOOGLE_API_KEY')      # personal AI Studio key (free tier)
LLM_MODEL = os.getenv("LLM_MODEL", "gemini-3.5-flash-lite")

# This client is used for query routing (route_query), RAG synthesis
# (rag_formatted_response), sub-query splitting (sub_queries) and composition.
openaiclient = OpenAI(api_key=google_api_key, base_url=GEMINI_BASE_URL)


In [44]:
# ── Rate-limit safety: auto-retry on Gemini free-tier 429s ────────────────────
# Free tier = 15 requests/min for gemini-3.5-flash-lite. A "Run All" fires LLM calls
# in bursts (route → sub-queries → compose → RBAC self-check), which trips the quota.
# This wraps every openaiclient.chat.completions.create() call so a 429 sleeps for the
# exact delay Gemini reports and then retries — the notebook pauses instead of crashing.
import time, re
from openai import RateLimitError

if not getattr(openaiclient.chat.completions, "_retry_wrapped", False):
    _orig_create = openaiclient.chat.completions.create

    def _create_with_retry(*args, **kwargs):
        for attempt in range(8):
            try:
                return _orig_create(*args, **kwargs)
            except RateLimitError as e:
                delay = 15.0
                m = re.search(r"retry in ([\d.]+)s", str(e))
                if m:
                    delay = float(m.group(1)) + 1
                print(f"[rate-limit] waiting {delay:.1f}s (attempt {attempt + 1}/8)...")
                time.sleep(delay)
        return _orig_create(*args, **kwargs)  # final attempt: let it raise

    openaiclient.chat.completions.create = _create_with_retry
    openaiclient.chat.completions._retry_wrapped = True
    print("Rate-limit retry wrapper installed on openaiclient.")


Rate-limit retry wrapper installed on openaiclient.


In [45]:
from openai import OpenAIError

def route_query(user_query: str):
    router_system_prompt =f"""
    As a professional query router, your objective is to correctly classify user input into one of three categories based on the source most relevant for answering the query:
    1. "OPENAI_QUERY": If the user's query appears to be answerable using information from OpenAI's official documentation about Agents, tools, models, APIs, or services (e.g., guardrails, agents, what is an agent, embeddings, moderation API, usage guidelines).
    2. "10K_DOCUMENT_QUERY": If the user's query pertains to a collection of documents from the 10k annual reports, datasets, or other structured documents, typically for research, analysis, or financial content.
    3. "INTERNET_QUERY": If the query is neither related to OpenAI nor the 10k documents specifically, or if the information might require a broader search (e.g., news, trends, tools outside these platforms), route it here.

    Your decision should be made by assessing the domain of the query.

    Always respond in this valid JSON format:
    {{
        "action": "OPENAI_QUERY" or "10K_DOCUMENT_QUERY" or "INTERNET_QUERY",
        "reason": "brief justification",
        "answer": "AT MAX 5 words answer. Leave empty if INTERNET_QUERY"
    }}

    EXAMPLES:

    - User: "How to fine-tune GPT-3?"
    Response:
    {{
        "action": "OPENAI_QUERY",
        "reason": "Fine-tuning is OpenAI-specific",
        "answer": "Use fine-tuning API"
    }}

    - User: "Where can I find the latest financial reports for the last 10 years?"
    Response:
    {{
        "action": "10K_DOCUMENT_QUERY",
        "reason": "Query related to annual reports",
        "answer": "Access through document database"
    }}

    - User: "Top leadership styles in 2024"
    Response:
    {{
        "action": "INTERNET_QUERY",
        "reason": "Needs current leadership trends",
        "answer": ""
    }}

    - User: "What's the difference between ChatGPT and Claude?"
    Response:
    {{
        "action": "INTERNET_QUERY",
        "reason": "Cross-comparison of different providers",
        "answer": ""
    }}

    Strictly follow this format for every query, and never deviate.
    User: {user_query}
    """

    try:
        # Query the routing model with the router prompt and user input.
        # Sent as a "user" message: Gemini's OpenAI-compat endpoint rejects a lone "system" one.
        response = openaiclient.chat.completions.create(
            model=LLM_MODEL,
            messages=[{"role": "user", "content": router_system_prompt}]
        )

        # Extract and parse the model's JSON response
        task_response = response.choices[0].message.content
        json_match = re.search(r"\{.*\}", task_response, re.DOTALL)
        json_text = json_match.group()
        parsed_response = json.loads(json_text)
        return parsed_response

    # Handle OpenAI API errors (e.g., rate limits, authentication)
    except OpenAIError as api_err:
        return {
            "action": "INTERNET_QUERY",
            "reason": f"OpenAI API error: {api_err}",
            "answer": ""
        }

    # Handle case where model response isn't valid JSON
    except json.JSONDecodeError as json_err:
        return {
            "action": "INTERNET_QUERY",
            "reason": f"JSON parsing error: {json_err}",
            "answer": ""
        }

    # Catch-all for any other unforeseen issues
    except Exception as err:
        return {
            "action": "INTERNET_QUERY",
            "reason": f"Unexpected error: {err}",
            "answer": ""
        }

In [46]:
route_query("what is the revenue of uber in 2021?")


{'action': '10K_DOCUMENT_QUERY',
 'reason': 'Query is about financial data from an annual report',
 'answer': 'Check Uber annual report'}

In [47]:
route_query("what is an AI Agent?")

{'action': 'OPENAI_QUERY',
 'reason': 'AI Agent concept covered in OpenAI docs',
 'answer': 'Autonomous task-executing system'}

## 3. Setting Up Qdrant Vector Database for Agentic RAG
In this step, we are connecting our agent to a pre-built vector database using Qdrant—a tool used to store and search document embeddings (numerical representations of text).

What Are We Doing?
We are loading an existing Qdrant database that was downloaded from a GitHub repository. This database already contains:

- Vectorized OpenAI documentation
- Vectorized 10-K financial filings

By loading this saved data:

- We save time (no need to re-embed the documents)
- We enable fast similarity search to retrieve relevant text chunks

This setup allows our system to perform semantic search, meaning it can understand the meaning of the user query and match it with the most relevant pieces of information stored in the database.


### Why This Matters in Agentic RAG
Once the router decides that the query should go to the OpenAI docs or the 10-K reports, our system uses Qdrant to:

- Search for the most relevant pieces of text
- Pass those to the model to generate a grounded answer

So, this step is essential to support retrieval-augmented generation (RAG) within our agentic flow.

#Data Sources:

**10K Database: Lyft 2024 & Uber 2021 SEC filings**

**OpenAI Docs: Official OpenAI documentation about Agents**

For lecture demo purposes, the vecitr database has already been created and hosted on Github which we will clone here. In order to create your own embeddings, the notebook and data will be hosted and shared on github

In [48]:
# Colab only — wipe a previous clone if you need a clean copy
if IN_COLAB:
    !rm -rf /content/multi-agent-course

In [49]:
# The prebuilt Qdrant collections (10-K + OpenAI docs) ship with the repo.
# On Colab we clone to get them; locally you already have them.
if IN_COLAB:
    !git clone https://github.com/hamzafarooq/multi-agent-course.git

Cloning into 'multi-agent-course'...
remote: Enumerating objects: 2823, done.
remote: Counting objects: 100% (434/434), done.
remote: Compressing objects: 100% (208/208), done.
remote: Total 2823 (delta 255), reused 265 (delta 223), pack-reused 2389 (from 2)
Receiving objects: 100% (2823/2823), 124.00 MiB | 18.24 MiB/s, done.
Resolving deltas: 100% (1069/1069), done.
Updating files: 100% (985/985), done.


In [50]:
# 🗄️ Initializing Qdrant client with the local path to the vector database
# Prebuilt collections (10-K and OpenAI docs) — cloned on Colab, already present locally.
import os

_MODULE = "modules/Module_3_Production_Agentic_RAG_AI_Systems"
if IN_COLAB:
    QDRANT_PATH = f"/content/multi-agent-course/{_MODULE}/Agentic_RAG/qdrant_data"
else:
    # the notebook lives in the module folder, so the data sits right next to it
    QDRANT_PATH = os.path.join(os.getcwd(), "Agentic_RAG", "qdrant_data")

print("Qdrant path:", QDRANT_PATH)
client = qdrant_client.AsyncQdrantClient(path=QDRANT_PATH)

Qdrant path: /content/multi-agent-course/modules/Module_3_Production_Agentic_RAG_AI_Systems/Agentic_RAG/qdrant_data


## 4. Building the Retriever and RAG for Vector Databases
In this section, we build the core logic that allows our agent to find relevant documents and generate grounded answers using them.

###Step 1: Import the Embedding Model
We start by importing the nomic-ai/nomic-embed-text-v1.5 model from Hugging Face. This model is used to convert any text (such as a user query) into a dense vector, known as an embedding. These embeddings capture the semantic meaning of text, allowing us to later compare and retrieve similar documents.


In [51]:
# Load the tokenizer and embedding model from Hugging Face
# This model converts raw text into dense vector representations (embeddings)
# Used for similarity search in Qdrant during document retrieval
text_tokenizer = AutoTokenizer.from_pretrained("nomic-ai/nomic-embed-text-v1.5", trust_remote_code=True)
text_model = AutoModel.from_pretrained("nomic-ai/nomic-embed-text-v1.5", trust_remote_code=True)

def get_text_embeddings(text):
    """
    Converts input text into a dense embedding using the Nomic embedding model.
    These embeddings are used to query Qdrant for semantically relevant document chunks.

    Args:
        text (str): The input text or query from the user.

    Returns:
        np.ndarray: A fixed-size vector representing the semantic meaning of the input.
    """
    # Tokenize and prepare input for the model
    inputs = text_tokenizer(text, return_tensors="pt", padding=True, truncation=True)

    # Forward pass to get model outputs
    outputs = text_model(**inputs)

    # Take the mean across all token embeddings to get a single vector (pooled representation)
    embeddings = outputs.last_hidden_state.mean(dim=1)

    # Convert to NumPy array and detach from computation graph
    return embeddings[0].detach().numpy()

# Example usage: Generate and preview the embedding of a test sentence
text = "This is a test sentence."
embeddings = get_text_embeddings(text)
print(embeddings[:5])  # Print first 5 dimensions for inspection


[ 1.2799693   0.40158385 -3.5162656  -0.39813167  1.5919145 ]


### Step 2: Define the Embedding Function
We then define a function get_text_embeddings() which:

- Tokenizes the input text
- Runs it through the model
- Computes the average of all token embeddings
- Returns a single vector that represents the full sentence

This vector will be used to query Qdrant to find the most relevant document chunks based on similarity.

In [52]:
def rag_formatted_response(user_query: str, context: list):
    """
    Generate a response to the user query using the provided context,
    with article references formatted as [1][2], etc.

    This function performs the final step in the RAG pipeline—synthesizing an answer
    from retrieved document chunks (context). It prompts the model to generate a
    grounded response, explicitly citing sources using a reference format.

    Args:
        user_query (str): The user's original question.
        context (list): List of text chunks retrieved from Qdrant (10-K or OpenAI docs).

    Returns:
        str: A generated response grounded in the retrieved context, with numbered citations.
    """

    # Construct a RAG prompt that includes both:
    # 1. The user's query
    # 2. The supporting context documents
    # The prompt instructs the model to answer using only the provided context,
    # and to include citations like [1], [2], etc. based on chunk IDs or order.
    rag_prompt = f"""
       Based on the given context, answer the user query: {user_query}\nContext:\n{context}
       and employ references to the ID of articles provided [ID], ensuring their relevance to the query.
       The referencing should always be in the format of [1][2]... etc. </instructions>
    """

    # Call the LLM to generate the response using the RAG-style prompt.
    # Sent as a "user" message (Gemini's OpenAI-compat endpoint rejects a lone "system" one).
    response = openaiclient.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {"role": "user", "content": rag_prompt},
        ]
    )

    # Return the model's generated answer
    return response.choices[0].message.content

### Step 3: Define the RAG Response Generator
After retrieving relevant text chunks from Qdrant, we use the rag_formatted_response() function to generate a final answer. This function:

- Takes the user query and the retrieved document chunks
- Builds a prompt that asks the language model (GPT-5.6-Luna) to answer the question using only the provided context
- Instructs the model to include references like [1], [2] for traceability

This ensures the output is not only informative but also grounded in actual retrieved data.

Together, these two functions lay the foundation for combining retrieval (from vector DB) and generation (from LLM) — the two pillars of a RAG system.



In [53]:
async def retrieve_and_response(user_query: str, action: str):
    """
    Retrieves relevant text chunks from the appropriate Qdrant collection
    based on the query type, then generates a response using RAG.

    This function powers the retrieval and response generation pipeline
    for queries that are classified as either OPENAI-related or 10-K related.
    It uses semantic search to fetch relevant context from a Qdrant vector store
    and then generates a response using that context via a RAG prompt.

    Args:
        user_query (str): The user's input question.
        action (str): The classification label from the router (e.g., "OPENAI_QUERY", "10K_DOCUMENT_QUERY").

    Returns:
        str: A model-generated response grounded in retrieved documents, or an error message.
    """

    # Define mapping of routing labels to their respective Qdrant collections
    collections = {
        "OPENAI_QUERY": "opnai_data",           # Collection of OpenAI documentation embeddings
        "10K_DOCUMENT_QUERY": "10k_data"        # Collection of 10-K financial document embeddings
    }

    try:
        # Ensure that the provided action is valid
        if action not in collections:
            return "Invalid action type for retrieval."

        # Step 1: Convert the user query into a dense vector (embedding)
        try:
            query = get_text_embeddings(user_query)
        except Exception as embed_err:
            return f"Embedding error: {embed_err}"  # Fail early if embedding fails

        # Step 2: Retrieve top-matching chunks from the relevant Qdrant collection
        try:
            text_hits = await client.query_points(
                collection_name=collections[action],  # Choose the right collection based on routing
                query=query,                          # The embedding of the user's query
                limit=3                               # Fetch top 3 relevant chunks
            )
        except Exception as qdrant_err:
            return f"Vector DB query error: {qdrant_err}"  # Handle Qdrant access issues

        # Extract the raw content from the retrieved vector hits
        contents = [point.payload['content'] for point in text_hits.points]

        # If no relevant content is found, return early
        if not contents:
            return "No relevant content found in the database."

        # Step 3: Pass the retrieved context to the RAG model to generate a response
        try:
            response = rag_formatted_response(user_query, contents)
            return response
        except Exception as rag_err:
            return f"RAG response error: {rag_err}"  # Handle generation failures

    # Catch any unforeseen errors in the overall process
    except Exception as err:
        return f"Unexpected error: {err}"


# 5. Putting It All Together: Running the Agentic RAG
In this final step, we combine everything into a single function that controls the entire Agentic RAG workflow. The agentic_rag() function acts as the main orchestrator of the system.

Here’s what it does:

- Prints the user's query for reference.
- Uses the router function (powered by GPT) to decide which type of data source to use:
  - OpenAI documentation
  - 10-K financial reports
- Internet search
- Calls the correct function based on the route:
- If it’s an OpenAI or 10-K query, it retrieves data from Qdrant and generates a RAG response.
- If it’s an Internet query, it uses the ARES API to fetch live information.
- Displays the final response, neatly formatted in the console.

This step brings the agentic loop full circle—from understanding the question, reasoning about where to search, to finally responding with the best possible answer.

In [54]:
# Dictionary that maps the route labels (decided by the router) to their respective functions
# Each type of query is handled differently:
# - OPENAI_QUERY and 10K_DOCUMENT_QUERY use document retrieval + RAG
# - INTERNET_QUERY uses a web search API
routes = {
    "OPENAI_QUERY": retrieve_and_response,
    "10K_DOCUMENT_QUERY": retrieve_and_response,
    "INTERNET_QUERY": get_internet_content,
}

def agentic_rag(user_query: str):
    """
    Main function that runs the full Agentic RAG system.

    This function takes a user's question, decides what type of query it is (OpenAI-related,
    financial document-related, or general internet), and then calls the right function
    to handle it. Finally, it prints out the full conversation and response.

    Args:
        user_query (str): The user's input question.

    Returns:
        None (It just prints the result nicely to the console)
    """

    #  Terminal color codes to make the printed output easier to read and visually structured
    CYAN = "\033[96m"
    GREY = "\033[90m"
    BOLD = "\033[1m"
    RESET = "\033[0m"

    try:
        # Step 1: Print the user's original question to the console
        print(f"{BOLD}{CYAN}👤 User Query:{RESET} {user_query}\n")

        # Step 2: Use the router (powered by GPT) to decide which route the query belongs to
        try:
            response = route_query(user_query)
        except Exception as route_err:
            # If something goes wrong while classifying the query, show an error message
            print(f"{BOLD}{CYAN}🤖 BOT RESPONSE:{RESET}\n")
            print(f"Routing error: {route_err}\n")
            return

        # Extract the routing decision and the reason behind it
        action = response.get("action")  # e.g., "OPENAI_QUERY"
        reason = response.get("reason")  # e.g., "Related to OpenAI tools"

        # Step 3: Show the selected route and why it was chosen
        print(f"{GREY}📍 Selected Route: {action}")
        print(f"📝 Reason: {reason}")
        print(f"⚙️ Processing query...{RESET}\n")

        # Step 4: Call the correct function depending on the route (retrieval or web search)
        try:
            route_function = routes.get(action)  # Find the function to use for this route
            if route_function:
                if action in ["OPENAI_QUERY", "10K_DOCUMENT_QUERY"]:
                    # Use asyncio.run for async functions, nest_asyncio will handle nested loops
                    result = asyncio.run(route_function(user_query, action))
                else:
                    # Otherwise, call it directly (e.g., get_internet_content is synchronous)
                    result = route_function(user_query, action)
            else:
                result = f"Unsupported action: {action}"  # Catch unknown routing types
        except Exception as exec_err:
            result = f"Execution error: {exec_err}"  # Handle failure in the chosen route function

        # Step 5: Print the final response to the user
        print(f"{BOLD}{CYAN}🤖 BOT RESPONSE:{RESET}\n")
        print(f"{result}\n")

    except Exception as err:
        # Catch-all for any unexpected errors in the overall logic
        print(f"{BOLD}{CYAN}🤖 BOT RESPONSE:{RESET}\n")
        print(f"Unexpected error occurred: {err}\n")


In [55]:
agentic_rag("what was uber revenue in 2021?")

👤 User Query: what was uber revenue in 2021?

📍 Selected Route: 10K_DOCUMENT_QUERY
📝 Reason: Query asks for financial data from annual reports
⚙️ Processing query...

🤖 BOT RESPONSE:

Based on the provided context, Uber's revenue in 2021 was $17.5 billion (or $17,455 million) [1][2].



In [56]:
agentic_rag("what was lyft revenue in 2022?")

👤 User Query: what was lyft revenue in 2022?

📍 Selected Route: 10K_DOCUMENT_QUERY
📝 Reason: Query asks for financial data from annual reports
⚙️ Processing query...

🤖 BOT RESPONSE:

Based on the provided context, Lyft's revenue in 2022 was $4,095,135 thousand (or approximately $4.095 billion) [2].



In [57]:
agentic_rag("List me down new LLMs in 2025")

👤 User Query: List me down new LLMs in 2025

📍 Selected Route: INTERNET_QUERY
📝 Reason: Requires recent external information on new models
⚙️ Processing query...

Getting your response from the internet 🌐 ...
🤖 BOT RESPONSE:

[1] LLM Tools Cheat Sheet 2025 | Manas Dasgupta
    LLM Tools Cheat Sheet 2025 A quick-reference guide for developers, builders, and AI enthusiasts. Streamline your AI stack with the best ...
    Source: https://www.linkedin.com/posts/manasdasgupta_llm-tools-cheat-sheet-2025-a-quick-reference-activity-7356524736165654529-c2dC

[2] LLM Deep Dive 2025: Why Claude 4 and GPT-5.1 Change ...
    The LLM landscape in late 2025 is a dynamic ecosystem, far removed from the nascent days of early generative AI. We're seeing a relentless ...
    Source: https://dev.to/dataformathub/llm-deep-dive-2025-why-claude-4-and-gpt-51-change-everything-4c26

[3] Top List of 16 Large Language Models: Comparing in 2026
    If you're looking for a list of large language models, this guide 

In [58]:
agentic_rag("how to work with chat completions?")

👤 User Query: how to work with chat completions?

📍 Selected Route: OPENAI_QUERY
📝 Reason: Chat completions are an OpenAI API feature
⚙️ Processing query...

🤖 BOT RESPONSE:

Based on the provided context, there is no information explaining how to work with chat completions. The text focuses instead on building agents, configuring instructions, orchestration, handoffs, and guardrails using the Agents SDK.



In [59]:
agentic_rag("best ways to build Agents")

👤 User Query: best ways to build Agents

📍 Selected Route: OPENAI_QUERY
📝 Reason: Query is about building AI agents, covered in OpenAI documentation
⚙️ Processing query...

🤖 BOT RESPONSE:

Based on the provided context, the best ways and foundational practices to build agents include:

* **Understand the Core Components:** An agent fundamentally consists of three core components: a **Model** (the LLM powering reasoning and decision-making), **Tools** (external functions or APIs to take action), and **Instructions** (explicit guidelines and guardrails defining behavior) [2].
* **Follow a Model Selection Strategy:** 
  * Build your agent prototype using the most capable model for every task to establish a performance baseline [2].
  * Focus on meeting your accuracy targets with the best models available first [2].
  * Optimize for cost and latency later by swapping in smaller, faster models for simpler tasks (like retrieval or intent classification) where they still achieve acceptable r

## 6. Role-Based Access Control (RBAC)

Everything so far assumes one kind of user: whoever asks gets whatever the router
picks. In a real deployment that's rarely true. An engineer shouldn't be able to pull
finance's 10-K numbers out of the vector store, and a finance analyst has no business
reading internal engineering docs — even though both are talking to the same agent.

RBAC puts a **permission check between the router's decision and the tool call**. The
router still reasons about *where* the answer lives; RBAC decides whether *this user*
is allowed to go there. If not, the request is rejected before any embedding, vector
search, or grounding call happens.

**This demo — 2 roles, 3 knowledge sources:**

| Knowledge source | Route label | `engineer` | `finance_analyst` |
|---|---|---|---|
| 📘 OpenAI documentation (Qdrant) | `OPENAI_QUERY` | ✅ | ✅ |
| 📗 10-K filings (Qdrant) | `10K_DOCUMENT_QUERY` | ❌ | ✅ |
| 🌐 Live internet search (SerpApi) | `INTERNET_QUERY` | ✅ | ❌ |

The two Qdrant collections and the SerpApi tool are the same ones built above — RBAC
is a layer on top, not a different pipeline.


In [60]:
# ── Users → role ─────────────────────────────────────────────────────────────
# Stand-in for a real identity provider. In production this comes from SSO/JWT
# claims or an internal users table — never a dict in the notebook.
USERS = {
    "alice": "engineer",
    "bob":   "finance_analyst",
}

# ── Roles → the route labels each role may reach ─────────────────────────────
# This is an allow-list: anything not listed here is denied by default.
ROLE_PERMISSIONS = {
    "engineer":        {"OPENAI_QUERY", "INTERNET_QUERY"},
    "finance_analyst": {"OPENAI_QUERY", "10K_DOCUMENT_QUERY"},
}

# Human-readable names, used only for clearer denial messages
SOURCE_LABELS = {
    "OPENAI_QUERY":       "OpenAI documentation",
    "10K_DOCUMENT_QUERY": "10-K financial filings",
    "INTERNET_QUERY":     "live internet search",
}


def has_access(user_id: str, action: str) -> bool:
    """True only if this user's role is explicitly allowed to use this route."""
    role = USERS.get(user_id)
    return role is not None and action in ROLE_PERMISSIONS.get(role, set())


def allowed_sources(user_id: str) -> set:
    """Every route label this user may reach — useful for constraining the router."""
    return ROLE_PERMISSIONS.get(USERS.get(user_id), set())


print("alice  (engineer)        →", allowed_sources("alice"))
print("bob    (finance_analyst) →", allowed_sources("bob"))
print("carol  (unknown user)    →", allowed_sources("carol"))

alice  (engineer)        → {'INTERNET_QUERY', 'OPENAI_QUERY'}
bob    (finance_analyst) → {'10K_DOCUMENT_QUERY', 'OPENAI_QUERY'}
carol  (unknown user)    → set()


In [61]:
def secure_agentic_rag(user_id: str, user_query: str):
    """
    The same agentic RAG loop as above, with one addition: after the router picks a
    route, the user's role must permit that route before the tool is called.

    Order of operations:
        1. Identify the user  → unknown users are rejected immediately
        2. Route the query    → router decides which knowledge source fits
        3. RBAC check         → role allowed to use that source? deny if not
        4. Retrieve + answer  → only ever reached by an authorized request

    Args:
        user_id (str): Who is asking (looked up in USERS).
        user_query (str): The question.

    Returns:
        str: The answer, or a denial message.
    """
    CYAN, GREY, RED, GREEN, BOLD, RESET = (
        "\033[96m", "\033[90m", "\033[91m", "\033[92m", "\033[1m", "\033[0m"
    )

    role = USERS.get(user_id)
    print(f"{BOLD}{CYAN}👤 User:{RESET} {user_id}  (role: {role or 'UNKNOWN'})")
    print(f"{BOLD}{CYAN}❓ Query:{RESET} {user_query}\n")

    # Step 1 — unknown identity is denied before anything else runs
    if role is None:
        print(f"{RED}🚫 ACCESS DENIED{RESET} — unknown user '{user_id}'.\n")
        return f"🚫 Access denied: unknown user '{user_id}'."

    # Step 2 — the router still does the reasoning about where the answer lives
    try:
        decision = route_query(user_query)
    except Exception as route_err:
        return f"Routing error: {route_err}"

    action = decision.get("action")
    reason = decision.get("reason")
    print(f"{GREY}📍 Selected Route: {action}")
    print(f"📝 Reason: {reason}{RESET}\n")

    # Step 3 — the gate. Nothing is embedded, searched, or grounded past this point
    #          unless the role is permitted to use the chosen source.
    if not has_access(user_id, action):
        source = SOURCE_LABELS.get(action, action)
        print(f"{RED}🚫 ACCESS DENIED{RESET} — role '{role}' may not query {source}.\n")
        return (
            f"🚫 Access denied: your role ('{role}') does not have permission to "
            f"query {source}."
        )

    print(f"{GREEN}✅ Access granted{RESET} — processing...\n")

    # Step 4 — identical to agentic_rag() from Section 5
    try:
        route_function = routes.get(action)
        if not route_function:
            return f"Unsupported action: {action}"
        if action in ["OPENAI_QUERY", "10K_DOCUMENT_QUERY"]:
            result = asyncio.run(route_function(user_query, action))
        else:
            result = route_function(user_query, action)
    except Exception as exec_err:
        result = f"Execution error: {exec_err}"

    print(f"{BOLD}{CYAN}🤖 BOT RESPONSE:{RESET}\n")
    print(f"{result}\n")
    return result

### Demo — same question, different roles

Each pair below sends the *identical* query as `alice` (engineer) and `bob`
(finance_analyst). The router makes the same decision both times — only the
permission check differs.


In [62]:
print("=" * 70)
print("1) alice (engineer) asks about the 10-K — FINANCE-ONLY → DENIED")
print("=" * 70)
secure_agentic_rag("alice", "what was uber revenue in 2021?")

1) alice (engineer) asks about the 10-K — FINANCE-ONLY → DENIED
👤 User: alice  (role: engineer)
❓ Query: what was uber revenue in 2021?

📍 Selected Route: 10K_DOCUMENT_QUERY
📝 Reason: Query asks for financial data from annual reports

🚫 ACCESS DENIED — role 'engineer' may not query 10-K financial filings.



"🚫 Access denied: your role ('engineer') does not have permission to query 10-K financial filings."

In [63]:
print("=" * 70)
print("2) bob (finance_analyst) asks the same question → ALLOWED")
print("=" * 70)
secure_agentic_rag("bob", "what was uber revenue in 2021?")

2) bob (finance_analyst) asks the same question → ALLOWED
👤 User: bob  (role: finance_analyst)
❓ Query: what was uber revenue in 2021?

📍 Selected Route: 10K_DOCUMENT_QUERY
📝 Reason: Query is about financial data from corporate reports

✅ Access granted — processing...

🤖 BOT RESPONSE:

Based on the provided context, Uber's revenue in 2021 was $17.5 billion (or $17,455 million) [1][2].



"Based on the provided context, Uber's revenue in 2021 was $17.5 billion (or $17,455 million) [1][2]."

In [64]:
print("=" * 70)
print("3) bob (finance_analyst) asks for live web results — ENGINEER-ONLY → DENIED")
print("=" * 70)
secure_agentic_rag("bob", "List me down new LLMs in 2025")

3) bob (finance_analyst) asks for live web results — ENGINEER-ONLY → DENIED
👤 User: bob  (role: finance_analyst)
❓ Query: List me down new LLMs in 2025

📍 Selected Route: INTERNET_QUERY
📝 Reason: Requires recent external information about LLMs

🚫 ACCESS DENIED — role 'finance_analyst' may not query live internet search.



"🚫 Access denied: your role ('finance_analyst') does not have permission to query live internet search."

In [65]:
print("=" * 70)
print("4) alice (engineer) asks about OpenAI docs — SHARED → ALLOWED")
print("=" * 70)
secure_agentic_rag("alice", "best ways to build Agents")

4) alice (engineer) asks about OpenAI docs — SHARED → ALLOWED
👤 User: alice  (role: engineer)
❓ Query: best ways to build Agents

📍 Selected Route: OPENAI_QUERY
📝 Reason: Building agents is covered in OpenAI documentation

✅ Access granted — processing...

🤖 BOT RESPONSE:

Based on the provided context, the best ways to build agents involve establishing core design foundations, selecting models effectively, configuring clear instructions, and utilizing appropriate tools. Here are the key practices:

### 1. Understand Agent Design Foundations
An agent fundamentally consists of three core components [2]:
* **Model:** An LLM to power reasoning and decision-making [2].
* **Tools:** External functions or APIs the agent uses to take action [2]. (If the number of required tools increases, consider splitting tasks across multiple agents) [3].
* **Instructions:** Explicit guidelines and guardrails defining agent behavior [2].

### 2. Model Selection Strategy
Different models involve tradeoffs r

'Based on the provided context, the best ways to build agents involve establishing core design foundations, selecting models effectively, configuring clear instructions, and utilizing appropriate tools. Here are the key practices:\n\n### 1. Understand Agent Design Foundations\nAn agent fundamentally consists of three core components [2]:\n* **Model:** An LLM to power reasoning and decision-making [2].\n* **Tools:** External functions or APIs the agent uses to take action [2]. (If the number of required tools increases, consider splitting tasks across multiple agents) [3].\n* **Instructions:** Explicit guidelines and guardrails defining agent behavior [2].\n\n### 2. Model Selection Strategy\nDifferent models involve tradeoffs related to task complexity, latency, and cost [2]. You can optimize your model usage by following these principles [2]:\n* Start by building your prototype with the most capable model for every task to establish a performance baseline (set up evals to measure this)

**Where this is still weak — and how you'd harden it:**

- **The router runs before the check.** One LLM call is spent classifying a query the
  user may not be allowed to ask. That's cheap and leaks nothing, but you can do
  better: pass `allowed_sources(user_id)` into the router prompt so it only ever
  chooses from routes the role can reach, and deny anything that falls outside.
- **This gates whole sources, not chunks.** When one collection mixes content that
  different roles may only *partially* see, push the check into the vector store with
  **payload-based filters** (Qdrant supports this natively) so restricted chunks never
  enter the retrieved context in the first place. You'll do exactly this, at file
  granularity, in `003. Agentic Router_semantic_caching_rbac.ipynb`.
- **Caching and RBAC interact badly if you're careless.** A shared semantic cache
  keyed only on the question will happily serve `bob`'s finance answer to `alice`.
  Any cache sitting behind an access check must be partitioned by role (or by the
  permitted source set) — think about this before you add one.
- **Every check is an audit point.** Log who asked for what and whether it was
  allowed; that trail is what makes the system defensible in a regulated environment.


# Assignment

**Required:** Part 1 — sub-query division. This is the graded piece for this notebook.

**Bonus (optional, ungraded):** RBAC with a semantic cache. It extends Section 6 and is a
useful warm-up for **ARGUS**, where you build multi-source retrieval with a real caching
layer and have to report cost with and without the cache.

| | Task | Status | Builds on |
|---|---|---|---|
| **Part 1** | Sub-query division | **Required** | Sections 2 & 5 |
| **Bonus** | RBAC + semantic cache, without cross-role leakage | Optional | Section 6 |

**Deliverable:** this notebook, run end to end, with Part 1 implemented in the stub cell.
If you take the bonus, include it in the same notebook with the self-check passing.


---

## Part 1 — Sub-query division

Right now a compound question is treated as one search. Ask *"What was Uber's revenue
in 2021 and what was Lyft's in 2024?"* and the router picks a single route and fires a
single retrieval — so you get a partial answer, or a muddled one.

Your job: break compound queries into focused sub-queries, run each one through the
full agentic pipeline independently, then compose a single coherent answer.

**Requirements**

1. Write `agentic_rag_multi(user_query)` that:
   - calls `sub_queries()` to split the query (reference implementation below),
   - **routes each sub-query separately** — they may legitimately land on different
     sources (one on `10K_DOCUMENT_QUERY`, another on `INTERNET_QUERY`),
   - collects the per-sub-query answers and synthesises **one** final response,
   - preserves citations from each sub-answer in the composed output.
2. Handle the single-question case without regression — one question in, one route,
   no extra LLM calls beyond the split.
3. Parse the model's JSON defensively. `sub_queries()` returns a *string*; it can come
   back wrapped in prose or a code fence. Don't let a malformed split crash the agent —
   fall back to treating the input as one query.

**Check yourself against these**

| Query | Expected behaviour |
|---|---|
| `"what was uber revenue in 2021?"` | 1 sub-query, 1 route, same as `agentic_rag()` |
| `"what was lyft revenue in 2021 and what was uber revenue in 2021"` | 2 sub-queries, both `10K_DOCUMENT_QUERY` |
| `"what was uber's 2021 revenue and what are the newest LLMs?"` | 2 sub-queries, **different** routes |

**Stretch:** run the sub-queries concurrently with `asyncio.gather` instead of in
sequence, and compare wall-clock time.


In [66]:
#Reference Code for sub query division (For Guidance Only)

def sub_queries(user_query):
  sub_queries_prompt= f"""
  You are a query router. If the input contains multiple distinct questions, break it into sub-questions. Otherwise, keep it as one. Return a JSON object like:

  {{
      "subQuestions": ["..."]
  }}


  Query: "{user_query}"
  Output:
  """
  # Sent as a "user" message (Gemini's OpenAI-compat endpoint rejects a lone "system" one).
  response = openaiclient.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {"role": "user", "content": sub_queries_prompt},
        ]
    )
  return response.choices[0].message.content

In [67]:
print(sub_queries("what was lyft revenue in 2021 and what was uber revenue in 2021"))

{
    "subQuestions": [
        "what was lyft revenue in 2021",
        "what was uber revenue in 2021"
    ]
}


In [68]:
# ── Part 1: sub-query division ───────────────────────────────────────────────
# Reuses the pipeline built above unchanged: sub_queries() (split), route_query()
# (routing), and the `routes` dispatch table (retrieve_and_response / get_internet_content).
# The only new logic is (a) a defensive parser for the split, and (b) a final compose step
# that stitches the independently-grounded sub-answers into one, keeping their citations.

def parse_sub_queries(user_query: str) -> list:
    """
    Call sub_queries() and parse its JSON *defensively*.

    sub_queries() returns a raw string that can arrive as clean JSON, wrapped in a
    ```json code fence, or padded with prose. We pull out the first {...} block, load it,
    and keep only non-empty string sub-questions. ANY failure (no JSON, bad JSON, empty
    list) falls back to treating the whole input as a single query — a malformed split
    must never crash the agent (Requirement 3).

    Returns:
        list[str]: one or more sub-questions; always at least [user_query].
    """
    try:
        raw = sub_queries(user_query)                      # the one allowed extra LLM call
        match = re.search(r"\{.*\}", raw, re.DOTALL)       # tolerate code fences / prose
        if not match:
            return [user_query]
        data = json.loads(match.group())
        subs = [s.strip() for s in data.get("subQuestions", [])
                if isinstance(s, str) and s.strip()]
        return subs if subs else [user_query]              # empty split → single query
    except Exception:
        return [user_query]                                # never let the split crash us


def answer_sub_query(sub_query: str) -> dict:
    """
    Route ONE sub-query and answer it through the existing pipeline.

    Identical dispatch to agentic_rag(): route_query() picks the source, then the matching
    function in `routes` runs it (vector routes are async → asyncio.run, like Section 5).

    Returns:
        dict: {"query", "action", "reason", "answer"} for this sub-query.
    """
    decision = route_query(sub_query)
    action = decision.get("action")
    reason = decision.get("reason")

    route_function = routes.get(action)
    try:
        if not route_function:
            answer = f"Unsupported action: {action}"
        elif action in ("OPENAI_QUERY", "10K_DOCUMENT_QUERY"):
            answer = asyncio.run(route_function(sub_query, action))   # async vector routes
        else:
            answer = route_function(sub_query, action)                # sync SerpApi route
    except Exception as exec_err:
        answer = f"Execution error: {exec_err}"

    return {"query": sub_query, "action": action, "reason": reason, "answer": answer}


def compose_answer(user_query: str, parts: list) -> str:
    """
    Synthesise ONE answer from several independently-grounded sub-answers.

    Each sub-answer already carries its own [1][2] citations that point at *its own*
    retrieved context. We must not renumber or merge those across sub-answers (that would
    silently mis-attribute a claim), so the prompt tells the model to keep every citation
    marker verbatim, glued to the claim it supports (Requirement: preserve citations).
    """
    sections = "\n\n".join(
        f"Sub-question {i} — routed to {p['action']}:\n{p['query']}\nGrounded answer:\n{p['answer']}"
        for i, p in enumerate(parts, start=1)
    )

    compose_prompt = f"""
    You are composing a single, coherent answer to a compound question from several
    sub-answers. Each sub-answer was retrieved and grounded independently and already
    contains its own bracketed citations like [1][2] that refer only to that sub-answer's
    own sources.

    Original question: {user_query}

    Sub-answers:
    {sections}

    Write one well-structured answer that addresses every sub-question. Keep each
    sub-answer's citation markers EXACTLY as written, next to the claim they support — do
    not renumber, merge, or invent citations, and do not add facts not present above.
    """

    # Sent as a "user" message (Gemini's OpenAI-compat endpoint rejects a lone "system" one).
    response = openaiclient.chat.completions.create(
        model=LLM_MODEL,                      # same client/model the rest of the notebook uses
        messages=[{"role": "user", "content": compose_prompt}],
    )
    return response.choices[0].message.content


def agentic_rag_multi(user_query: str):
    """
    Split a compound query, route + answer each sub-query independently, then compose one
    cited answer.

    Single-question case has no regression: the split returns one sub-query, we route and
    answer it exactly once, and we skip the compose call — so the only cost beyond
    agentic_rag() is the single sub_queries() split call (Requirement 2 & 4).
    """
    CYAN, GREY, BOLD, RESET = "\033[96m", "\033[90m", "\033[1m", "\033[0m"
    print(f"{BOLD}{CYAN}👤 User Query:{RESET} {user_query}\n")

    # 1. Split (defensively)
    subs = parse_sub_queries(user_query)
    plural = "y" if len(subs) == 1 else "ies"
    print(f"{GREY}🔀 Split into {len(subs)} sub-quer{plural}:")
    for i, s in enumerate(subs, start=1):
        print(f"   {i}. {s}")
    print(RESET)

    # 2. Route + answer each sub-query independently (they may hit different sources)
    parts = []
    for i, s in enumerate(subs, start=1):
        p = answer_sub_query(s)
        parts.append(p)
        print(f"{GREY}📍 Sub-query {i} → {p['action']}  ({p['reason']}){RESET}")

    # 3. Compose one answer — but not for a lone sub-query (no extra LLM call, no regression)
    result = parts[0]["answer"] if len(parts) == 1 else compose_answer(user_query, parts)

    print(f"\n{BOLD}{CYAN}🤖 BOT RESPONSE:{RESET}\n")
    print(f"{result}\n")
    return result


# Test cases from the table above
agentic_rag_multi("what was uber revenue in 2021?")                                   # 1 sub-query, 1 route
agentic_rag_multi("what was lyft revenue in 2021 and what was uber revenue in 2021")  # 2 sub-queries, both 10-K
agentic_rag_multi("what was uber's 2021 revenue and what are the newest LLMs?")       # 2 sub-queries, different routes

👤 User Query: what was uber revenue in 2021?

🔀 Split into 1 sub-query:
   1. what was uber revenue in 2021?

📍 Sub-query 1 → 10K_DOCUMENT_QUERY  (Query asks for corporate financial data found in annual 10-K reports)

🤖 BOT RESPONSE:

Based on the provided context, Uber's revenue in 2021 was $17.5 billion (or specifically $17,455 million, as shown in the financial statements) [1][2].

👤 User Query: what was lyft revenue in 2021 and what was uber revenue in 2021

🔀 Split into 2 sub-queries:
   1. what was lyft revenue in 2021
   2. what was uber revenue in 2021

📍 Sub-query 1 → 10K_DOCUMENT_QUERY  (Query asks for corporate financial data from an annual report)
📍 Sub-query 2 → 10K_DOCUMENT_QUERY  (Query asks for financial data found in annual 10K documents)

🤖 BOT RESPONSE:

In 2021, Lyft's total revenue was listed alongside reporting periods with a figure of **$4,095,135** (in thousands) [1]. Meanwhile, Uber's revenue in 2021 was $17.5 billion (or $17,455 million) [1][2].

👤 User Query:

"Uber's revenue for 2021 was $17.5 billion (specifically $17,455 million) [1][2]. \n\nRegarding the newest Large Language Models (LLMs), recent developments highlight advanced open-source models and coding-specific models (such as Qwen2.5-Coder-32B by Alibaba) [2], alongside ongoing quality evaluations and benchmarks testing modern models on complex tasks like difficult math problems [5], which significantly outperform older iterations like Llama 2 [3]."

In [69]:
# ── Bonus: your implementation ───────────────────────────────────────────────
# Reuse USERS / ROLE_PERMISSIONS / has_access / SOURCE_LABELS from Section 6.
!pip install -q faiss-cpu          # not installed by the Setup cell — only the bonus needs it

import faiss
import numpy as np

# Answers whose truth changes with time must never be cached (a stock price cached for an
# hour is a wrong answer served fast). Same idea as is_time_sensitive() in notebook 003.
_TIME_SENSITIVE_KEYWORDS = [
    "today", "tonight", "now", "currently", "current", "latest", "recent", "recently",
    "this week", "this month", "this year", "yesterday", "tomorrow", "last week",
    "upcoming", "live", "breaking", "stock price", "share price", "weather", "forecast",
    "real-time", "realtime", "as of now", "right now",
]

def is_time_sensitive(question: str) -> bool:
    """True if the answer would go stale — such queries bypass the cache entirely."""
    q = question.lower()
    return any(kw in q for kw in _TIME_SENSITIVE_KEYWORDS)


class RoleAwareSemanticCache:
    """
    A semantic cache that cannot serve an answer across a permission boundary.

    Design choice — PARTITIONED, keyed by SOURCE (the route label), not by user or role.

      RBAC in secure_agentic_rag() gates by *source* (OPENAI_QUERY / 10K_DOCUMENT_QUERY /
      INTERNET_QUERY). So the source is exactly the permission boundary — I give each source
      its own FAISS index and only ever search the index for the source the caller was just
      authorized to use. Two consequences:

        • Leak-safe by construction. A lookup for a source only runs *after* the RBAC gate
          confirms this caller may use that source (see the call order below). alice (engineer)
          asking a finance question is denied at the gate and never reaches the 10-K index, so
          she can't be semantically matched into bob's finance rows — not even by paraphrase.

        • No wasted recompute on shared sources. OPENAI_QUERY is allowed for BOTH roles; a
          per-*role* cache would answer it once per role. Keying on the source caches it once
          and safely serves every role permitted to that source. (This is the efficiency point
          from the assignment hint: "caching per permitted source instead of per role fixes it.")

      Role changes: since entries are keyed on source, not on the user, a user who *loses*
      access to a source simply stops passing the gate for it — their old entries live under
      the source partition and remain correctly unreachable to them. Nothing to invalidate.

    Embeddings reuse the notebook's local Nomic get_text_embeddings() (no second model load);
    vectors are L2-normalized so the squared-L2 threshold behaves like a cosine cutoff
    (threshold 0.2 ≈ cosine ≥ 0.9).
    """

    def __init__(self, threshold: float = 0.2):
        self.embedding_dim = 768
        self.threshold = threshold                 # max squared-L2 distance for a hit
        # partition (source label) -> {"index": faiss index, "answers": [str], "questions": [str]}
        self.partitions: dict = {}

    def _embed(self, question: str) -> np.ndarray:
        """Nomic embedding, L2-normalized, shaped (1, dim) float32 for FAISS."""
        v = np.asarray(get_text_embeddings(question), dtype="float32")
        v = v / (np.linalg.norm(v) + 1e-12)
        return v.reshape(1, -1)

    def check(self, partition: str, question: str):
        """
        Look for a near-enough question WITHIN one source partition only.

        Returns (hit: bool, answer: str | None, embedding, similarity: float | None).
        The embedding is returned so a miss can reuse it in add() without re-encoding.
        """
        embedding = self._embed(question)
        part = self.partitions.get(partition)
        if not part or part["index"].ntotal == 0:
            return False, None, embedding, None

        D, I = part["index"].search(embedding, 1)
        if I[0][0] != -1 and D[0][0] <= self.threshold:
            row = int(I[0][0])
            return True, part["answers"][row], embedding, float(1.0 - D[0][0])
        return False, None, embedding, None

    def add(self, partition: str, question: str, answer: str, embedding: np.ndarray):
        """Store an answer under its source partition (never crosses the boundary)."""
        part = self.partitions.get(partition)
        if part is None:
            part = {"index": faiss.IndexFlatL2(self.embedding_dim),
                    "answers": [], "questions": []}
            self.partitions[partition] = part
        part["index"].add(embedding)
        part["answers"].append(answer)
        part["questions"].append(question)


def _run_secure_pipeline(user_query: str, action: str) -> str:
    """Run the authorized route through the existing pipeline (same dispatch as Section 5/6)."""
    route_function = routes.get(action)
    if not route_function:
        return f"Unsupported action: {action}"
    try:
        if action in ("OPENAI_QUERY", "10K_DOCUMENT_QUERY"):
            return asyncio.run(route_function(user_query, action))   # async vector routes
        return route_function(user_query, action)                    # sync SerpApi route
    except Exception as exec_err:
        return f"Execution error: {exec_err}"


def secure_agentic_rag_cached(user_id: str, user_query: str, cache) -> dict:
    """
    RBAC-gated agentic RAG with a role-aware (source-partitioned) semantic cache.

    Order of operations (this order is the whole point — it's what prevents the leak):
        unknown user      → DENIED   (no route, no embedding, no cache, no LLM)
        route the query   → which source does this need?
        role not allowed  → DENIED   (still NO cache lookup — a denial must not be cacheable)
        time-sensitive    → run live, return MISS, do NOT store
        cache lookup      → HIT  → return stored answer
                          → MISS → run pipeline, store under the source partition, return

    Returns:
        dict: {"answer": str, "status": "HIT" | "MISS" | "DENIED", "role": str | None}
    """
    role = USERS.get(user_id)

    # 1. Identity — unknown users are rejected before ANY work (no LLM, no embed, no cache).
    if role is None:
        return {"answer": f"🚫 Access denied: unknown user '{user_id}'.",
                "status": "DENIED", "role": None}

    # 2. Route — the router decides which source the answer lives in.
    action = route_query(user_query).get("action")

    # 3. RBAC gate — deny BEFORE any cache lookup, so a denial is never cacheable and a
    #    forbidden question can never be semantically matched into another role's rows.
    if not has_access(user_id, action):
        source = SOURCE_LABELS.get(action, action)
        return {"answer": f"🚫 Access denied: role '{role}' may not query {source}.",
                "status": "DENIED", "role": role}

    # 4. Time-sensitive answers are never cached — always compute fresh.
    if is_time_sensitive(user_query):
        return {"answer": _run_secure_pipeline(user_query, action),
                "status": "MISS", "role": role}

    # 5. Cache lookup — scoped to the (authorized) source partition only.
    hit, cached_answer, embedding, _ = cache.check(action, user_query)
    if hit:
        return {"answer": cached_answer, "status": "HIT", "role": role}

    # 6. Miss — run the pipeline, store under the source partition, return.
    answer = _run_secure_pipeline(user_query, action)
    cache.add(action, user_query, answer, embedding)
    return {"answer": answer, "status": "MISS", "role": role}

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 24.8 MB/s eta 0:00:00


In [70]:
# ── Bonus: self-check — this must pass ───────────────────────────────────────
# It asserts behaviour, not wording, so your answer text can be anything.

def run_self_check():
    cache = RoleAwareSemanticCache()
    q_fin = "what was uber revenue in 2021?"
    q_doc = "how do I build an agent with the OpenAI Agents SDK?"

    # 1. bob may read financials — first ask is a MISS
    r = secure_agentic_rag_cached("bob", q_fin, cache)
    assert r["status"] == "MISS", f"expected MISS, got {r['status']}"

    # 2. bob asks again — served from cache
    r = secure_agentic_rag_cached("bob", q_fin, cache)
    assert r["status"] == "HIT", f"expected HIT, got {r['status']}"

    # 3. THE LEAK TEST — alice must be denied, never served bob's cached answer
    r = secure_agentic_rag_cached("alice", q_fin, cache)
    assert r["status"] == "DENIED", f"LEAK: alice got {r['status']} on finance data"

    # 4. a near-paraphrase must also be denied, not semantically matched into bob's rows
    r = secure_agentic_rag_cached("alice", "how much revenue did Uber make in 2021?", cache)
    assert r["status"] == "DENIED", f"LEAK: alice got {r['status']} via paraphrase"

    # 5. unknown users are rejected outright
    r = secure_agentic_rag_cached("carol", q_doc, cache)
    assert r["status"] == "DENIED", f"expected DENIED for unknown user, got {r['status']}"

    # 6. a shared source still caches normally within a role
    assert secure_agentic_rag_cached("alice", q_doc, cache)["status"] == "MISS"
    assert secure_agentic_rag_cached("alice", q_doc, cache)["status"] == "HIT"

    print("✅ All checks passed — cache is fast and does not leak across roles.")


run_self_check()

✅ All checks passed — cache is fast and does not leak across roles.
